# 04 — CTGAN Synthetic Data Generation & Validation

## Frozen pipeline position

This notebook takes the validated 10,000-row seed dataset from Notebook 03 and creates a larger synthetic dataset using CTGAN.

**Seed dataset → CTGAN → synthetic data → distribution validation → relationship validation → logical checks → final synthetic dataset**

### Important boundary

CTGAN learns the statistical structure of the synthetic seed population. It does **not** make the data real-world observations, and CTGAN output must not be described as real patient/participant data.

The final dataset is for prototype ML development only.


In [2]:
%pip install ctgan

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.1 MB 1.5 MB/s eta 0:00:01
   -------------------- ------------------- 1.0/2.1 MB 1.5 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.1 MB 1.5 MB/s eta 0:00:01
   ------------------------------ --------- 1.6/2.1 MB 1.6 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.3 MB/s  0:00:01
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/122.1 MB 1.7 MB/s eta 0:01:12
   ---------------------------------------- 1.0/122.1 MB 1.9 MB/s eta 0:01:04
   ---------------------------------------

In [3]:
# ============================================================
# 0. INSTALLATION CHECK
# ============================================================

import sys
import subprocess

try:
    from ctgan import CTGAN
    print("ctgan package: available")
except ImportError:
    print("ctgan package not found.")
    print("Install once with:")
    print("%pip install ctgan")
    raise

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from scipy.stats import ks_2samp

np.random.seed(42)

print("Python:", sys.version)


ctgan package: available
Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


## 1. Load the validated seed dataset

In [4]:
SEED_PATH = "../data/processed/seed_risk_dataset.csv"

seed = pd.read_csv(SEED_PATH, encoding="latin1")

print("Seed shape:", seed.shape)
display(seed.head())


Seed shape: (10000, 40)


,STATE/UT,spousal_violence_pct,pregnancy_violence_pct,sexual_violence_pct,age,violence_escalation,threat_to_kill,violent_jealousy,recent_separation,lethal_weapon,...,family_supports_abuse,perpetrator_present,safe_now,can_leave_safely,medical_help,contact_requested,immediate_threat,dawi_reference_score,severe_ipv_risk,immediate_safety_risk
0,Jammu & Kashmir,5.9,0.3,1.4,19,0,0,0,0,0,...,0,0,0,1,0,0,0,6,0,MEDIUM
1,Uttarakhand,12.5,2.4,0.0,22,0,0,0,1,0,...,0,0,1,1,0,1,0,6,0,LOW
2,Puducherry,29.8,1.3,0.0,34,1,0,0,0,0,...,0,0,1,1,0,0,0,4,0,LOW
3,Meghalaya,23.2,2.5,7.9,22,1,0,0,0,0,...,0,0,1,1,0,0,0,2,0,LOW
4,Chandigarh,9.7,0.0,NaN,40,0,0,0,0,0,...,0,0,1,1,0,1,0,2,0,LOW


In [5]:
print("Immediate safety distribution:")
display(seed["immediate_safety_risk"].value_counts(normalize=True))

print("\nSevere IPV reference distribution:")
display(seed["severe_ipv_risk"].value_counts(normalize=True))

print("\nMissing values:")
print(seed.isna().sum().sum())


Immediate safety distribution:


immediate_safety_risk
MEDIUM    0.5181
LOW       0.4222
HIGH      0.0597
Name: proportion, dtype: float64


Severe IPV reference distribution:


severe_ipv_risk
0    0.9324
1    0.0676
Name: proportion, dtype: float64


Missing values:
252


## 2. Define the CTGAN training columns

We exclude the state-level NFHS prevalence anchor columns from CTGAN training because they are contextual reference variables rather than individual-level observations.

`STATE/UT` remains because geography is useful for preserving contextual variation.

The CTGAN table therefore contains:
- age
- 26 DA-WI factors
- immediate-safety variables
- DA-WI reference score
- prototype risk labels
- state/UT


In [6]:
EXCLUDE_FROM_CTGAN = [
    "spousal_violence_pct",
    "pregnancy_violence_pct",
    "sexual_violence_pct"
]

ctgan_df = seed.drop(
    columns=EXCLUDE_FROM_CTGAN
).copy()

print("CTGAN input shape:", ctgan_df.shape)
print("Excluded reference columns:", EXCLUDE_FROM_CTGAN)


CTGAN input shape: (10000, 37)
Excluded reference columns: ['spousal_violence_pct', 'pregnancy_violence_pct', 'sexual_violence_pct']


## 3. Prepare categorical columns

CTGAN can directly model discrete columns when they are passed through `discrete_columns`.

We keep binary variables as integers and treat the risk label as categorical.


In [7]:
discrete_columns = [
    "STATE/UT",
    "violence_escalation",
    "threat_to_kill",
    "violent_jealousy",
    "recent_separation",
    "lethal_weapon",
    "avoids_arrest",
    "strangulation",
    "illegal_drug_use",
    "problem_drinking",
    "violence_during_pregnancy",
    "partner_capable_of_killing",
    "suicide_threat_attempt",
    "withholds_necessities",
    "intimidating_behavior",
    "rumors",
    "false_accusations",
    "family_rejection",
    "social_isolation",
    "inlaws_support_abuse",
    "infertility_related_abuse",
    "healthcare_neglect",
    "family_honor_threat",
    "leaving_threat",
    "hide_abuse",
    "lack_of_support",
    "family_supports_abuse",
    "perpetrator_present",
    "safe_now",
    "can_leave_safely",
    "medical_help",
    "contact_requested",
    "immediate_threat",
    "severe_ipv_risk",
    "immediate_safety_risk"
]

missing_discrete = [
    c for c in discrete_columns
    if c not in ctgan_df.columns
]

print("Missing discrete columns:", missing_discrete)
print("Discrete columns:", len(discrete_columns))


Missing discrete columns: []
Discrete columns: 35


## 4. Train CTGAN

For the prototype, we start with a moderate configuration that is practical on a normal laptop.

If training is slow, reduce `epochs` to 100. For the final experiment, 200 epochs is the default starting point.

CTGAN is stochastic, so `random_state=42` is used for reproducibility where supported by the installed version.


In [8]:
CTGAN_EPOCHS = 200

model = CTGAN(
    epochs=CTGAN_EPOCHS,
    batch_size=500,
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    verbose=True
)

print("Training CTGAN...")
model.fit(
    ctgan_df,
    discrete_columns=discrete_columns
)

print("CTGAN training complete.")


Training CTGAN...


Gen. (-08.15) | Discrim. (-00.02): 100%|█████████████████████████████████████████████| 200/200 [15:24<00:00,  4.62s/it]

CTGAN training complete.


## 5. Generate synthetic records

In [9]:
N_SYNTHETIC = 20000

synthetic = model.sample(N_SYNTHETIC)

print("Synthetic shape:", synthetic.shape)
display(synthetic.head())


Synthetic shape: (20000, 37)


,STATE/UT,age,violence_escalation,threat_to_kill,violent_jealousy,recent_separation,lethal_weapon,avoids_arrest,strangulation,illegal_drug_use,...,family_supports_abuse,perpetrator_present,safe_now,can_leave_safely,medical_help,contact_requested,immediate_threat,dawi_reference_score,severe_ipv_risk,immediate_safety_risk
0,Andaman & Nicobar Islands,38,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,1,4,0,MEDIUM
1,Kerala,39,0,0,0,0,0,0,0,0,...,0,0,0,1,1,1,0,5,0,MEDIUM
2,Madhya Pradesh,27,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,3,0,MEDIUM
3,Nagaland,41,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,11,0,MEDIUM
4,Karnataka,34,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,0,MEDIUM


In [10]:
print("Synthetic dtypes:")
display(synthetic.dtypes)

print("\nSynthetic missing values:")
print(synthetic.isna().sum().sum())

print("\nSynthetic immediate-risk distribution:")
display(
    synthetic["immediate_safety_risk"]
    .value_counts(normalize=True)
)


Synthetic dtypes:


STATE/UT                      object
age                            int64
violence_escalation            int64
threat_to_kill                 int64
violent_jealousy               int64
recent_separation              int64
lethal_weapon                  int64
avoids_arrest                  int64
strangulation                  int64
illegal_drug_use               int64
problem_drinking               int64
violence_during_pregnancy      int64
partner_capable_of_killing     int64
suicide_threat_attempt         int64
withholds_necessities          int64
intimidating_behavior          int64
rumors                         int64
false_accusations              int64
family_rejection               int64
social_isolation               int64
inlaws_support_abuse           int64
infertility_related_abuse      int64
healthcare_neglect             int64
family_honor_threat            int64
leaving_threat                 int64
hide_abuse                     int64
lack_of_support                int64
f


Synthetic missing values:
0

Synthetic immediate-risk distribution:


immediate_safety_risk
MEDIUM    0.56665
LOW       0.36670
HIGH      0.06665
Name: proportion, dtype: float64

## 6. Normalize generated binary columns

CTGAN can occasionally represent binary values as numeric variants depending on the installed version and learned distribution. We normalize them back to 0/1 for downstream validation.

We do **not** silently round continuous variables.


In [11]:
binary_columns = [
    "violence_escalation",
    "threat_to_kill",
    "violent_jealousy",
    "recent_separation",
    "lethal_weapon",
    "avoids_arrest",
    "strangulation",
    "illegal_drug_use",
    "problem_drinking",
    "violence_during_pregnancy",
    "partner_capable_of_killing",
    "suicide_threat_attempt",
    "withholds_necessities",
    "intimidating_behavior",
    "rumors",
    "false_accusations",
    "family_rejection",
    "social_isolation",
    "inlaws_support_abuse",
    "infertility_related_abuse",
    "healthcare_neglect",
    "family_honor_threat",
    "leaving_threat",
    "hide_abuse",
    "lack_of_support",
    "family_supports_abuse",
    "perpetrator_present",
    "safe_now",
    "can_leave_safely",
    "medical_help",
    "contact_requested",
    "immediate_threat",
    "severe_ipv_risk"
]

for col in binary_columns:
    synthetic[col] = (
        pd.to_numeric(synthetic[col], errors="coerce")
        .round()
        .clip(0, 1)
        .astype("Int64")
    )

synthetic["age"] = pd.to_numeric(
    synthetic["age"],
    errors="coerce"
).round().clip(18, 49)

print("Binary normalization complete.")


Binary normalization complete.


## 7. Recalculate DA-WI score

The score is recomputed from the generated binary factors rather than trusting the CTGAN-generated score column. This lets us test whether the generated risk factors are internally consistent.


In [12]:
DAWI_WEIGHTS = dict(
    zip(dawi["feature"], dawi["weight"])
) if "dawi" in globals() else pd.read_csv(
    "../data/processed/dawi_risk_factors.csv"
).set_index("feature")["weight"].to_dict()

risk_columns = list(DAWI_WEIGHTS.keys())

synthetic["dawi_reference_score_recomputed"] = sum(
    synthetic[col].astype(int) * weight
    for col, weight in DAWI_WEIGHTS.items()
)

synthetic["dawi_score_difference"] = (
    pd.to_numeric(synthetic["dawi_reference_score"], errors="coerce")
    - synthetic["dawi_reference_score_recomputed"]
)

print(
    "Rows with inconsistent generated DA-WI score:",
    (synthetic["dawi_score_difference"].abs() > 0.5).sum()
)

display(
    synthetic[
        [
            "dawi_reference_score",
            "dawi_reference_score_recomputed",
            "dawi_score_difference"
        ]
    ].head()
)


Rows with inconsistent generated DA-WI score: 17903


,dawi_reference_score,dawi_reference_score_recomputed,dawi_score_difference
0,4,2,2
1,5,0,5
2,3,0,3
3,11,6,5
4,0,0,0


## 8. Distribution comparison

For each important binary factor we compare seed prevalence with synthetic prevalence.

A small difference is expected. Large differences indicate that CTGAN has not preserved the seed distribution adequately.


In [13]:
comparison_rows = []

for col in binary_columns:
    seed_rate = seed[col].mean()
    syn_rate = synthetic[col].astype(float).mean()

    comparison_rows.append({
        "feature": col,
        "seed_rate": seed_rate,
        "synthetic_rate": syn_rate,
        "absolute_difference": abs(seed_rate - syn_rate)
    })

distribution_comparison = pd.DataFrame(comparison_rows)

display(
    distribution_comparison
    .sort_values("absolute_difference", ascending=False)
)

print(
    "Mean absolute prevalence difference:",
    distribution_comparison["absolute_difference"].mean()
)

print(
    "Maximum absolute prevalence difference:",
    distribution_comparison["absolute_difference"].max()
)


,feature,seed_rate,synthetic_rate,absolute_difference
30,contact_requested,0.2505,0.46540,0.21490
28,can_leave_safely,0.6936,0.50145,0.19215
31,immediate_threat,0.1048,0.28030,0.17550
26,perpetrator_present,0.2244,0.09190,0.13250
2,violent_jealousy,0.0928,0.22155,0.12875
27,safe_now,0.7038,0.60050,0.10330
1,threat_to_kill,0.0546,0.15325,0.09865
13,intimidating_behavior,0.1281,0.04460,0.08350
25,family_supports_abuse,0.0385,0.10765,0.06915
7,illegal_drug_use,0.0385,0.10300,0.06450


Mean absolute prevalence difference: 0.05216363636363636
Maximum absolute prevalence difference: 0.21489999999999998


In [14]:
print("Risk-class comparison:")

risk_comparison = pd.DataFrame({
    "seed": seed["immediate_safety_risk"].value_counts(normalize=True),
    "synthetic": synthetic["immediate_safety_risk"].value_counts(normalize=True)
}).fillna(0)

display(risk_comparison)


Risk-class comparison:


,seed,synthetic
immediate_safety_risk,,
MEDIUM,0.5181,0.56665
LOW,0.4222,0.36670
HIGH,0.0597,0.06665


## 9. Continuous-variable comparison

We compare age and DA-WI reference score distributions using summary statistics and the Kolmogorov–Smirnov statistic.

The KS statistic is used as a diagnostic, not as a formal claim that the synthetic data are statistically identical.


In [15]:
continuous_columns = [
    "age",
    "dawi_reference_score"
]

continuous_rows = []

for col in continuous_columns:
    seed_values = pd.to_numeric(seed[col], errors="coerce").dropna()
    syn_values = pd.to_numeric(synthetic[col], errors="coerce").dropna()

    ks_stat, ks_p = ks_2samp(seed_values, syn_values)

    continuous_rows.append({
        "feature": col,
        "seed_mean": seed_values.mean(),
        "synthetic_mean": syn_values.mean(),
        "seed_std": seed_values.std(),
        "synthetic_std": syn_values.std(),
        "KS_statistic": ks_stat,
        "KS_p_value": ks_p
    })

continuous_comparison = pd.DataFrame(continuous_rows)

display(continuous_comparison)


,feature,seed_mean,synthetic_mean,seed_std,synthetic_std,KS_statistic,KS_p_value
0,age,31.2404,33.09605,7.535893,8.441776,0.15475,6.846445e-140
1,dawi_reference_score,3.6163,4.02150,3.054310,2.715844,0.21690,7.743080e-276


## 10. Relationship preservation

We compare pairwise correlations among the main binary DA-WI factors. CTGAN should preserve broad relationships rather than simply matching individual prevalences.


In [16]:
corr_seed = seed[risk_columns].corr()
corr_syn = synthetic[risk_columns].astype(float).corr()

corr_difference = (
    corr_seed - corr_syn
).abs()

print(
    "Mean absolute correlation difference:",
    corr_difference.values[np.triu_indices_from(corr_difference, k=1)].mean()
)

print(
    "Maximum absolute correlation difference:",
    corr_difference.values[np.triu_indices_from(corr_difference, k=1)].max()
)


Mean absolute correlation difference: 0.017611796458184465
Maximum absolute correlation difference: 0.15049172521786222


## 11. Logical consistency checks

These checks remove synthetic records that contradict the application's basic safety logic.

Rules:
1. `immediate_threat = 1` requires `safe_now = 0` and `perpetrator_present = 1`.
2. `HIGH` risk requires a high-risk current-safety condition.
3. `LOW` must not simultaneously contain an immediate threat.
4. DA-WI score must equal the recomputed score.


In [17]:
# Recalculate operational risk from the generated immediate-safety variables.

high_condition_syn = (
    (
        (synthetic["safe_now"] == 0)
        & (synthetic["perpetrator_present"] == 1)
        & (synthetic["can_leave_safely"] == 0)
    )
    |
    (
        (synthetic["medical_help"] == 1)
        & (synthetic["immediate_threat"] == 1)
    )
    |
    (
        (synthetic["lethal_weapon"] == 1)
        & (synthetic["immediate_threat"] == 1)
    )
)

medium_condition_syn = (
    (synthetic["safe_now"] == 0)
    |
    (synthetic["perpetrator_present"] == 1)
    |
    (synthetic["can_leave_safely"] == 0)
    |
    (synthetic["medical_help"] == 1)
)

expected_risk = np.select(
    [high_condition_syn, medium_condition_syn],
    ["HIGH", "MEDIUM"],
    default="LOW"
)

synthetic["expected_immediate_safety_risk"] = expected_risk

risk_mismatch = (
    synthetic["immediate_safety_risk"] !=
    synthetic["expected_immediate_safety_risk"]
).sum()

immediate_threat_invalid = (
    (synthetic["immediate_threat"] == 1)
    &
    ~(
        (synthetic["safe_now"] == 0)
        &
        (synthetic["perpetrator_present"] == 1)
    )
).sum()

score_invalid = (
    synthetic["dawi_score_difference"].abs() > 0.5
).sum()

print("Risk-label mismatches:", risk_mismatch)
print("Invalid immediate-threat records:", immediate_threat_invalid)
print("Invalid DA-WI scores:", score_invalid)


Risk-label mismatches: 9843
Invalid immediate-threat records: 5302
Invalid DA-WI scores: 17903


## 12. Repair invalid generated records

Synthetic generation is probabilistic. For a prototype dataset, logically invalid records should not be passed to model training.

We therefore:
- replace the operational risk label with the deterministic risk derived from its inputs;
- replace the DA-WI score with the recomputed score;
- remove records with missing binary values;
- keep a count of how many records required correction.


In [18]:
synthetic["immediate_safety_risk"] = (
    synthetic["expected_immediate_safety_risk"]
)

synthetic["dawi_reference_score"] = (
    synthetic["dawi_reference_score_recomputed"]
)

before = len(synthetic)

synthetic = synthetic.dropna(
    subset=binary_columns + [
        "age",
        "STATE/UT",
        "dawi_reference_score",
        "immediate_safety_risk"
    ]
).copy()

after = len(synthetic)

print("Rows before cleanup:", before)
print("Rows after cleanup:", after)
print("Rows removed:", before - after)


Rows before cleanup: 20000
Rows after cleanup: 20000
Rows removed: 0


## 13. Final validation

In [19]:
print("=" * 70)
print("FINAL SYNTHETIC DATASET VALIDATION")
print("=" * 70)

print("Shape:", synthetic.shape)

print("\nRisk distribution:")
display(
    synthetic["immediate_safety_risk"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nSevere IPV distribution:")
display(
    synthetic["severe_ipv_risk"]
    .value_counts(normalize=True)
)

print("\nMissing values:", synthetic.isna().sum().sum())

print(
    "DA-WI score range:",
    synthetic["dawi_reference_score"].min(),
    "to",
    synthetic["dawi_reference_score"].max()
)

print(
    "Age range:",
    synthetic["age"].min(),
    "to",
    synthetic["age"].max()
)


FINAL SYNTHETIC DATASET VALIDATION
Shape: (20000, 40)

Risk distribution:


immediate_safety_risk
HIGH      0.05885
LOW       0.27950
MEDIUM    0.66165
Name: proportion, dtype: float64


Severe IPV distribution:


severe_ipv_risk
0    0.949
1    0.051
Name: proportion, dtype: Float64


Missing values: 0
DA-WI score range: 0 to 25
Age range: 18 to 49


## 14. Save final synthetic dataset

This file becomes the input for the next stage: model training.

The next stage will compare:
- Logistic Regression
- rule-engine override
- LOW/MEDIUM/HIGH mapping
- calibration/threshold selection
- model size and quantization suitability


In [20]:
FINAL_PATH = "../data/processed/synthetic_risk_dataset.csv"

synthetic.to_csv(
    FINAL_PATH,
    index=False
)

print("Saved:", FINAL_PATH)
print("Final shape:", synthetic.shape)


Saved: ../data/processed/synthetic_risk_dataset.csv
Final shape: (20000, 40)


## Frozen output

**Final file:**

`data/processed/synthetic_risk_dataset.csv`

Do not train the final model in this notebook.

Next stage:

**Synthetic dataset → Logistic Regression → rule-engine override → risk output → quantization → on-device inference.**


In [21]:
# ============================================================
# NOTEBOOK 04 — FINAL OUTPUT FOR MODEL TRAINING
# ============================================================

print("=" * 80)
print("1. FINAL DATASET")
print("=" * 80)

print("Shape:", synthetic.shape)
print("Saved:", os.path.exists(
    "../data/processed/synthetic_risk_dataset.csv"
))
print("Missing values:", synthetic.isna().sum().sum())


print("\n" + "=" * 80)
print("2. IMMEDIATE SAFETY RISK DISTRIBUTION")
print("=" * 80)

risk_dist = (
    synthetic["immediate_safety_risk"]
    .value_counts(normalize=True)
    .sort_index()
)

display(risk_dist)


print("\n" + "=" * 80)
print("3. SEVERE IPV DISTRIBUTION")
print("=" * 80)

display(
    synthetic["severe_ipv_risk"]
    .value_counts(normalize=True)
)


print("\n" + "=" * 80)
print("4. DA-WI SCORE")
print("=" * 80)

print(
    synthetic["dawi_reference_score"]
    .describe()
)

print(
    "\nDA-WI score by risk:"
)

display(
    synthetic.groupby("immediate_safety_risk")[
        "dawi_reference_score"
    ].agg(
        ["count", "mean", "median", "min", "max"]
    )
)


print("\n" + "=" * 80)
print("5. DISTRIBUTION VALIDATION")
print("=" * 80)

print(
    "Mean absolute prevalence difference:",
    distribution_comparison[
        "absolute_difference"
    ].mean()
)

print(
    "Maximum absolute prevalence difference:",
    distribution_comparison[
        "absolute_difference"
    ].max()
)

display(
    distribution_comparison.sort_values(
        "absolute_difference",
        ascending=False
    ).head(10)
)


print("\n" + "=" * 80)
print("6. CONTINUOUS VARIABLE VALIDATION")
print("=" * 80)

display(continuous_comparison)


print("\n" + "=" * 80)
print("7. RELATIONSHIP VALIDATION")
print("=" * 80)

upper_triangle = np.triu_indices_from(
    corr_difference,
    k=1
)

print(
    "Mean absolute correlation difference:",
    corr_difference.values[upper_triangle].mean()
)

print(
    "Maximum absolute correlation difference:",
    corr_difference.values[upper_triangle].max()
)


print("\n" + "=" * 80)
print("8. LOGICAL CONSISTENCY")
print("=" * 80)

print(
    "Risk-label mismatches after repair:",
    (
        synthetic["immediate_safety_risk"]
        != synthetic["expected_immediate_safety_risk"]
    ).sum()
)

print(
    "Invalid immediate-threat records:",
    (
        (synthetic["immediate_threat"] == 1)
        &
        ~(
            (synthetic["safe_now"] == 0)
            &
            (synthetic["perpetrator_present"] == 1)
        )
    ).sum()
)

print(
    "DA-WI score inconsistencies:",
    (
        synthetic["dawi_reference_score"]
        != synthetic["dawi_reference_score_recomputed"]
    ).sum()
)


print("\n" + "=" * 80)
print("9. FINAL SAMPLE")
print("=" * 80)

display(
    synthetic[
        [
            "age",
            "dawi_reference_score",
            "safe_now",
            "perpetrator_present",
            "can_leave_safely",
            "medical_help",
            "contact_requested",
            "immediate_threat",
            "severe_ipv_risk",
            "immediate_safety_risk"
        ]
    ].head(10)
)

1. FINAL DATASET
Shape: (20000, 40)
Saved: True
Missing values: 0

2. IMMEDIATE SAFETY RISK DISTRIBUTION


immediate_safety_risk
HIGH      0.05885
LOW       0.27950
MEDIUM    0.66165
Name: proportion, dtype: float64


3. SEVERE IPV DISTRIBUTION


severe_ipv_risk
0    0.949
1    0.051
Name: proportion, dtype: Float64


4. DA-WI SCORE
count    20000.000000
mean         4.065600
std          3.135712
min          0.000000
25%          2.000000
50%          4.000000
75%          6.000000
max         25.000000
Name: dawi_reference_score, dtype: float64

DA-WI score by risk:


...

,feature,seed_rate,synthetic_rate,absolute_difference
30,contact_requested,0.2505,0.46540,0.21490
28,can_leave_safely,0.6936,0.50145,0.19215
31,immediate_threat,0.1048,0.28030,0.17550
26,perpetrator_present,0.2244,0.09190,0.13250
2,violent_jealousy,0.0928,0.22155,0.12875
27,safe_now,0.7038,0.60050,0.10330
1,threat_to_kill,0.0546,0.15325,0.09865
13,intimidating_behavior,0.1281,0.04460,0.08350
25,family_supports_abuse,0.0385,0.10765,0.06915
7,illegal_drug_use,0.0385,0.10300,0.06450



6. CONTINUOUS VARIABLE VALIDATION


,feature,seed_mean,synthetic_mean,seed_std,synthetic_std,KS_statistic,KS_p_value
0,age,31.2404,33.09605,7.535893,8.441776,0.15475,6.846445e-140
1,dawi_reference_score,3.6163,4.02150,3.054310,2.715844,0.21690,7.743080e-276



7. RELATIONSHIP VALIDATION
Mean absolute correlation difference: 0.017611796458184465
Maximum absolute correlation difference: 0.15049172521786222

8. LOGICAL CONSISTENCY
Risk-label mismatches after repair: 0
Invalid immediate-threat records: 5302
DA-WI score inconsistencies: 0

9. FINAL SAMPLE


,age,dawi_reference_score,safe_now,perpetrator_present,can_leave_safely,medical_help,contact_requested,immediate_threat,severe_ipv_risk,immediate_safety_risk
0,38,2,0,0,1,0,0,1,0,MEDIUM
1,39,0,0,0,1,1,1,0,0,MEDIUM
2,27,0,0,0,1,0,0,0,0,MEDIUM
3,41,6,0,0,0,0,0,0,0,MEDIUM
4,34,0,1,0,1,0,0,0,0,LOW
5,39,8,0,0,1,0,0,0,0,MEDIUM
6,28,0,1,0,0,0,1,0,0,MEDIUM
7,40,0,1,0,1,0,0,0,0,LOW
8,38,5,1,0,1,0,1,1,0,LOW
9,36,8,1,0,0,0,0,1,0,MEDIUM


In [ ]:
#imporving the ctgan

In [22]:
# ============================================================
# REVISED CTGAN INPUT
# ============================================================

seed = pd.read_csv(
    "../data/processed/seed_risk_dataset.csv",
    encoding="latin1"
)

print("Seed:", seed.shape)

Seed: (10000, 40)


In [23]:
DERIVED_COLUMNS = [
    "dawi_reference_score",
    "severe_ipv_risk",
    "immediate_threat",
    "immediate_safety_risk"
]

REFERENCE_COLUMNS = [
    "spousal_violence_pct",
    "pregnancy_violence_pct",
    "sexual_violence_pct"
]

ctgan_df = seed.drop(
    columns=DERIVED_COLUMNS + REFERENCE_COLUMNS
).copy()

print("CTGAN input shape:", ctgan_df.shape)
print("Excluded derived:", DERIVED_COLUMNS)
print("Excluded reference:", REFERENCE_COLUMNS)

CTGAN input shape: (10000, 33)
Excluded derived: ['dawi_reference_score', 'severe_ipv_risk', 'immediate_threat', 'immediate_safety_risk']
Excluded reference: ['spousal_violence_pct', 'pregnancy_violence_pct', 'sexual_violence_pct']


In [24]:
discrete_columns = [
    "STATE/UT",
    "violence_escalation",
    "threat_to_kill",
    "violent_jealousy",
    "recent_separation",
    "lethal_weapon",
    "avoids_arrest",
    "strangulation",
    "illegal_drug_use",
    "problem_drinking",
    "violence_during_pregnancy",
    "partner_capable_of_killing",
    "suicide_threat_attempt",
    "withholds_necessities",
    "intimidating_behavior",
    "rumors",
    "false_accusations",
    "family_rejection",
    "social_isolation",
    "inlaws_support_abuse",
    "infertility_related_abuse",
    "healthcare_neglect",
    "family_honor_threat",
    "leaving_threat",
    "hide_abuse",
    "lack_of_support",
    "family_supports_abuse",
    "perpetrator_present",
    "safe_now",
    "can_leave_safely",
    "medical_help",
    "contact_requested"
]

print("Discrete columns:", len(discrete_columns))

Discrete columns: 32


In [25]:
from ctgan import CTGAN

model_revised = CTGAN(
    epochs=200,
    batch_size=500,
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    verbose=True
)

model_revised.fit(
    ctgan_df,
    discrete_columns=discrete_columns
)

print("Revised CTGAN training complete.")

Gen. (-07.59) | Discrim. (-00.25): 100%|█████████████████████████████████████████████| 200/200 [13:55<00:00,  4.18s/it]

Revised CTGAN training complete.


In [26]:
N_SYNTHETIC = 20000

synthetic_v2 = model_revised.sample(N_SYNTHETIC)

print("Synthetic V2 shape:", synthetic_v2.shape)
display(synthetic_v2.head())

Synthetic V2 shape: (20000, 33)


,STATE/UT,age,violence_escalation,threat_to_kill,violent_jealousy,recent_separation,lethal_weapon,avoids_arrest,strangulation,illegal_drug_use,...,family_honor_threat,leaving_threat,hide_abuse,lack_of_support,family_supports_abuse,perpetrator_present,safe_now,can_leave_safely,medical_help,contact_requested
0,Uttar Pradesh,29,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,0
1,NCT Delhi,39,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,1,0,1
2,Assam,36,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,West Bengal,24,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,Uttar Pradesh,18,0,0,0,0,0,0,1,1,...,0,0,0,0,0,0,1,1,0,0


In [27]:
binary_columns = [
    c for c in discrete_columns
    if c != "STATE/UT"
]

for col in binary_columns:
    synthetic_v2[col] = (
        pd.to_numeric(
            synthetic_v2[col],
            errors="coerce"
        )
        .round()
        .clip(0, 1)
        .astype("Int64")
    )

synthetic_v2["age"] = (
    pd.to_numeric(
        synthetic_v2["age"],
        errors="coerce"
    )
    .round()
    .clip(18, 49)
)

print("Normalization complete.")

Normalization complete.


In [29]:
# ============================================================
# RECALCULATE DA-WI SCORE
# ============================================================

# Load DA-WI weights from Notebook 02 output
DAWI_PATH = "../data/processed/dawi_risk_factors.csv"

dawi = pd.read_csv(
    DAWI_PATH,
    encoding="latin1"
)

print("DA-WI table:", dawi.shape)

DAWI_WEIGHTS = dict(
    zip(
        dawi["feature"],
        dawi["weight"]
    )
)

risk_columns = list(DAWI_WEIGHTS.keys())

print("DA-WI factors:", len(risk_columns))
print("Maximum theoretical DA-WI score:", sum(DAWI_WEIGHTS.values()))

# Recalculate score from the generated factors
synthetic_v2["dawi_reference_score"] = sum(
    synthetic_v2[col].astype(int) * weight
    for col, weight in DAWI_WEIGHTS.items()
)

print("\nGenerated DA-WI score:")
print(synthetic_v2["dawi_reference_score"].describe())

DA-WI table: (26, 7)
DA-WI factors: 26
Maximum theoretical DA-WI score: 64

Generated DA-WI score:
count    20000.000000
mean         2.977100
std          2.730009
min          0.000000
25%          0.000000
50%          2.000000
75%          5.000000
max         22.000000
Name: dawi_reference_score, dtype: float64


In [30]:
synthetic_v2["immediate_threat"] = (
    (synthetic_v2["safe_now"] == 0) &
    (synthetic_v2["perpetrator_present"] == 1)
).astype(int)

print(
    synthetic_v2["immediate_threat"].value_counts()
)

immediate_threat
0    18992
1     1008
Name: count, dtype: int64


In [31]:
high_condition = (
    (
        (synthetic_v2["safe_now"] == 0) &
        (synthetic_v2["perpetrator_present"] == 1) &
        (synthetic_v2["can_leave_safely"] == 0)
    )
    |
    (
        (synthetic_v2["medical_help"] == 1) &
        (synthetic_v2["immediate_threat"] == 1)
    )
    |
    (
        (synthetic_v2["lethal_weapon"] == 1) &
        (synthetic_v2["immediate_threat"] == 1)
    )
)

medium_condition = (
    (synthetic_v2["safe_now"] == 0)
    |
    (synthetic_v2["perpetrator_present"] == 1)
    |
    (synthetic_v2["can_leave_safely"] == 0)
    |
    (synthetic_v2["medical_help"] == 1)
)

synthetic_v2["immediate_safety_risk"] = np.select(
    [high_condition, medium_condition],
    ["HIGH", "MEDIUM"],
    default="LOW"
)

display(
    synthetic_v2["immediate_safety_risk"]
    .value_counts(normalize=True)
)

immediate_safety_risk
MEDIUM    0.72055
LOW       0.25720
HIGH      0.02225
Name: proportion, dtype: float64

In [32]:
score_norm = (
    synthetic_v2["dawi_reference_score"] / 64.0
)

severe_prob = np.clip(
    0.02
    + 0.65 * score_norm
    + 0.12 * synthetic_v2["threat_to_kill"]
    + 0.15 * synthetic_v2["strangulation"]
    + 0.10 * synthetic_v2["lethal_weapon"],
    0.001,
    0.98
)

synthetic_v2["severe_ipv_risk"] = (
    np.random.random(len(synthetic_v2))
    < severe_prob
).astype(int)

print(
    synthetic_v2["severe_ipv_risk"]
    .value_counts(normalize=True)
)

severe_ipv_risk
0    0.92935
1    0.07065
Name: proportion, dtype: float64


In [33]:
invalid_immediate = (
    (synthetic_v2["immediate_threat"] == 1)
    &
    ~(
        (synthetic_v2["safe_now"] == 0)
        &
        (synthetic_v2["perpetrator_present"] == 1)
    )
).sum()

print(
    "Invalid immediate-threat records:",
    invalid_immediate
)

print(
    "Missing values:",
    synthetic_v2.isna().sum().sum()
)

print(
    "DA-WI score range:",
    synthetic_v2["dawi_reference_score"].min(),
    "to",
    synthetic_v2["dawi_reference_score"].max()
)

print("\nRisk distribution:")
display(
    synthetic_v2["immediate_safety_risk"]
    .value_counts(normalize=True)
)

Invalid immediate-threat records: 0
Missing values: 0
DA-WI score range: 0 to 22

Risk distribution:


immediate_safety_risk
MEDIUM    0.72055
LOW       0.25720
HIGH      0.02225
Name: proportion, dtype: float64

In [34]:
comparison_v2 = []

for col in binary_columns:
    seed_rate = seed[col].mean()
    syn_rate = synthetic_v2[col].astype(float).mean()

    comparison_v2.append({
        "feature": col,
        "seed_rate": seed_rate,
        "synthetic_rate": syn_rate,
        "absolute_difference": abs(
            seed_rate - syn_rate
        )
    })

comparison_v2 = pd.DataFrame(comparison_v2)

print(
    "Mean absolute prevalence difference:",
    comparison_v2["absolute_difference"].mean()
)

print(
    "Maximum absolute prevalence difference:",
    comparison_v2["absolute_difference"].max()
)

display(
    comparison_v2.sort_values(
        "absolute_difference",
        ascending=False
    ).head(10)
)

Mean absolute prevalence difference: 0.043748387096774186
Maximum absolute prevalence difference: 0.15625


,feature,seed_rate,synthetic_rate,absolute_difference
0,violence_escalation,0.2648,0.10855,0.15625
27,safe_now,0.7038,0.55730,0.14650
28,can_leave_safely,0.6936,0.55445,0.13915
26,perpetrator_present,0.2244,0.11165,0.11275
15,false_accusations,0.0587,0.15730,0.09860
13,intimidating_behavior,0.1281,0.05790,0.07020
6,strangulation,0.0211,0.09055,0.06945
3,recent_separation,0.0771,0.14010,0.06300
24,lack_of_support,0.0943,0.03460,0.05970
17,social_isolation,0.0929,0.03695,0.05595


In [35]:
# ============================================================
# CONTROLLED RECONSTRUCTION — START
# ============================================================

# Work on a copy so synthetic_v2 remains available for comparison
final_synthetic = synthetic_v2.copy()

print("Starting rows:", len(final_synthetic))
print("Starting columns:", len(final_synthetic.columns))

Starting rows: 20000
Starting columns: 37


In [36]:
# ============================================================
# RECONSTRUCT RISK CONTEXT FROM DA-WI FACTORS
# ============================================================

# Weighted DA-WI score from the CTGAN-generated factors
final_synthetic["dawi_reference_score"] = sum(
    final_synthetic[col].astype(int) * weight
    for col, weight in DAWI_WEIGHTS.items()
)

# Normalize to 0-1
score_norm = (
    final_synthetic["dawi_reference_score"] / 64.0
).clip(0, 1)

# Use important immediate-danger factors to strengthen the
# current-risk context.
risk_context = (
    0.55 * score_norm
    + 0.20 * final_synthetic["violence_escalation"]
    + 0.15 * final_synthetic["threat_to_kill"]
    + 0.10 * final_synthetic["lethal_weapon"]
)

risk_context = risk_context.clip(0, 1)

print("Risk-context statistics:")
print(risk_context.describe())

Risk-context statistics:
count     20000.0
mean     0.056432
std      0.080986
min           0.0
25%           0.0
50%      0.025781
75%      0.051563
max      0.510156
dtype: Float64


In [37]:
# ============================================================
# APP VARIABLE 1 — PERPETRATOR PRESENT
# ============================================================

np.random.seed(100)

p_perpetrator = (
    0.08
    + 0.35 * risk_context
    + 0.10 * final_synthetic["violence_escalation"]
)

p_perpetrator = p_perpetrator.clip(0.02, 0.80)

final_synthetic["perpetrator_present"] = (
    np.random.random(len(final_synthetic))
    < p_perpetrator
).astype(int)

print(
    "Perpetrator present rate:",
    final_synthetic["perpetrator_present"].mean()
)

Perpetrator present rate: 0.11145


In [38]:
# ============================================================
# APP VARIABLE 2 — SAFE NOW
# ============================================================

np.random.seed(101)

p_safe = (
    0.92
    - 0.55 * risk_context
    - 0.25 * final_synthetic["perpetrator_present"]
)

p_safe = p_safe.clip(0.05, 0.95)

final_synthetic["safe_now"] = (
    np.random.random(len(final_synthetic))
    < p_safe
).astype(int)

print(
    "Safe now rate:",
    final_synthetic["safe_now"].mean()
)

Safe now rate: 0.8616


In [39]:
# ============================================================
# APP VARIABLE 3 — CAN LEAVE SAFELY
# ============================================================

np.random.seed(102)

p_leave = (
    0.90
    - 0.40 * risk_context
    - 0.30 * final_synthetic["perpetrator_present"]
    - 0.12 * final_synthetic["social_isolation"]
)

p_leave = p_leave.clip(0.05, 0.95)

final_synthetic["can_leave_safely"] = (
    np.random.random(len(final_synthetic))
    < p_leave
).astype(int)

print(
    "Can leave safely rate:",
    final_synthetic["can_leave_safely"].mean()
)

Can leave safely rate: 0.83605


In [40]:
# ============================================================
# APP VARIABLE 4 — MEDICAL HELP
# ============================================================

np.random.seed(103)

p_medical = (
    0.02
    + 0.10 * risk_context
    + 0.25 * final_synthetic["violence_during_pregnancy"]
    + 0.20 * final_synthetic["strangulation"]
    + 0.10 * final_synthetic["lethal_weapon"]
)

p_medical = p_medical.clip(0.01, 0.70)

final_synthetic["medical_help"] = (
    np.random.random(len(final_synthetic))
    < p_medical
).astype(int)

print(
    "Medical help rate:",
    final_synthetic["medical_help"].mean()
)

Medical help rate: 0.05085


In [41]:
# ============================================================
# APP VARIABLE 5 — CONTACT REQUESTED
# ============================================================

np.random.seed(104)

p_contact = (
    0.08
    + 0.20 * risk_context
    + 0.25 * (1 - final_synthetic["safe_now"])
    + 0.15 * final_synthetic["perpetrator_present"]
)

p_contact = p_contact.clip(0.02, 0.85)

final_synthetic["contact_requested"] = (
    np.random.random(len(final_synthetic))
    < p_contact
).astype(int)

print(
    "Contact requested rate:",
    final_synthetic["contact_requested"].mean()
)

Contact requested rate: 0.14525


In [42]:
# ============================================================
# DERIVED VARIABLE — IMMEDIATE THREAT
# ============================================================

final_synthetic["immediate_threat"] = (
    (final_synthetic["safe_now"] == 0)
    &
    (final_synthetic["perpetrator_present"] == 1)
).astype(int)

print(
    "Immediate threat rate:",
    final_synthetic["immediate_threat"].mean()
)

Immediate threat rate: 0.04245


In [43]:
# ============================================================
# DERIVED VARIABLE — DA-WI REFERENCE SCORE
# ============================================================

final_synthetic["dawi_reference_score"] = sum(
    final_synthetic[col].astype(int) * weight
    for col, weight in DAWI_WEIGHTS.items()
)

print(
    final_synthetic["dawi_reference_score"].describe()
)

count    20000.000000
mean         2.977100
std          2.730009
min          0.000000
25%          0.000000
50%          2.000000
75%          5.000000
max         22.000000
Name: dawi_reference_score, dtype: float64


In [44]:
# ============================================================
# DERIVED VARIABLE — IMMEDIATE SAFETY RISK
# ============================================================

high_condition = (
    (
        (final_synthetic["safe_now"] == 0)
        &
        (final_synthetic["perpetrator_present"] == 1)
        &
        (final_synthetic["can_leave_safely"] == 0)
    )
    |
    (
        (final_synthetic["medical_help"] == 1)
        &
        (final_synthetic["immediate_threat"] == 1)
    )
    |
    (
        (final_synthetic["lethal_weapon"] == 1)
        &
        (final_synthetic["immediate_threat"] == 1)
    )
)

medium_condition = (
    (final_synthetic["safe_now"] == 0)
    |
    (final_synthetic["perpetrator_present"] == 1)
    |
    (final_synthetic["can_leave_safely"] == 0)
    |
    (final_synthetic["medical_help"] == 1)
)

final_synthetic["immediate_safety_risk"] = np.select(
    [high_condition, medium_condition],
    ["HIGH", "MEDIUM"],
    default="LOW"
)

display(
    final_synthetic["immediate_safety_risk"]
    .value_counts(normalize=True)
)

immediate_safety_risk
LOW       0.65805
MEDIUM    0.32090
HIGH      0.02105
Name: proportion, dtype: float64

In [45]:
# ============================================================
# DERIVED VARIABLE — SEVERE IPV REFERENCE LABEL
# ============================================================

np.random.seed(105)

score_norm = (
    final_synthetic["dawi_reference_score"] / 64.0
).clip(0, 1)

severe_prob = np.clip(
    0.02
    + 0.65 * score_norm
    + 0.12 * final_synthetic["threat_to_kill"]
    + 0.15 * final_synthetic["strangulation"]
    + 0.10 * final_synthetic["lethal_weapon"],
    0.001,
    0.98
)

final_synthetic["severe_ipv_risk"] = (
    np.random.random(len(final_synthetic))
    < severe_prob
).astype(int)

print(
    final_synthetic["severe_ipv_risk"]
    .value_counts(normalize=True)
)

severe_ipv_risk
0    0.9265
1    0.0735
Name: proportion, dtype: float64


In [46]:
# ============================================================
# APP-FACING VARIABLE VALIDATION
# ============================================================

app_features = [
    "safe_now",
    "perpetrator_present",
    "can_leave_safely",
    "medical_help",
    "contact_requested"
]

app_comparison = []

for col in app_features:

    seed_rate = seed[col].mean()

    final_rate = (
        final_synthetic[col]
        .astype(float)
        .mean()
    )

    app_comparison.append({
        "feature": col,
        "seed_rate": seed_rate,
        "final_synthetic_rate": final_rate,
        "absolute_difference": abs(
            seed_rate - final_rate
        )
    })

app_comparison = pd.DataFrame(
    app_comparison
)

display(app_comparison)

print(
    "Mean app-feature difference:",
    app_comparison[
        "absolute_difference"
    ].mean()
)

print(
    "Maximum app-feature difference:",
    app_comparison[
        "absolute_difference"
    ].max()
)

,feature,seed_rate,final_synthetic_rate,absolute_difference
0,safe_now,0.7038,0.86160,0.15780
1,perpetrator_present,0.2244,0.11145,0.11295
2,can_leave_safely,0.6936,0.83605,0.14245
3,medical_help,0.0621,0.05085,0.01125
4,contact_requested,0.2505,0.14525,0.10525


Mean app-feature difference: 0.10594
Maximum app-feature difference: 0.15780000000000005


In [47]:
# ============================================================
# DA-WI FEATURE VALIDATION
# ============================================================

dawi_comparison = []

for col in risk_columns:

    seed_rate = seed[col].mean()

    final_rate = (
        final_synthetic[col]
        .astype(float)
        .mean()
    )

    dawi_comparison.append({
        "feature": col,
        "seed_rate": seed_rate,
        "final_synthetic_rate": final_rate,
        "absolute_difference": abs(
            seed_rate - final_rate
        )
    })

dawi_comparison = pd.DataFrame(
    dawi_comparison
)

display(
    dawi_comparison.sort_values(
        "absolute_difference",
        ascending=False
    ).head(15)
)

print(
    "Mean DA-WI prevalence difference:",
    dawi_comparison[
        "absolute_difference"
    ].mean()
)

print(
    "Maximum DA-WI prevalence difference:",
    dawi_comparison[
        "absolute_difference"
    ].max()
)

,feature,seed_rate,final_synthetic_rate,absolute_difference
0,violence_escalation,0.2648,0.10855,0.15625
15,false_accusations,0.0587,0.15730,0.09860
13,intimidating_behavior,0.1281,0.05790,0.07020
6,strangulation,0.0211,0.09055,0.06945
3,recent_separation,0.0771,0.14010,0.06300
24,lack_of_support,0.0943,0.03460,0.05970
17,social_isolation,0.0929,0.03695,0.05595
23,hide_abuse,0.0845,0.03130,0.05320
2,violent_jealousy,0.0928,0.04440,0.04840
12,withholds_necessities,0.0689,0.03090,0.03800


Mean DA-WI prevalence difference: 0.035475
Maximum DA-WI prevalence difference: 0.15625


In [48]:
# ============================================================
# FINAL LOGICAL CONSISTENCY
# ============================================================

invalid_immediate = (
    (final_synthetic["immediate_threat"] == 1)
    &
    ~(
        (final_synthetic["safe_now"] == 0)
        &
        (final_synthetic["perpetrator_present"] == 1)
    )
).sum()

# Recalculate expected risk independently
expected_high = (
    (
        (final_synthetic["safe_now"] == 0)
        &
        (final_synthetic["perpetrator_present"] == 1)
        &
        (final_synthetic["can_leave_safely"] == 0)
    )
    |
    (
        (final_synthetic["medical_help"] == 1)
        &
        (final_synthetic["immediate_threat"] == 1)
    )
    |
    (
        (final_synthetic["lethal_weapon"] == 1)
        &
        (final_synthetic["immediate_threat"] == 1)
    )
)

expected_medium = (
    (final_synthetic["safe_now"] == 0)
    |
    (final_synthetic["perpetrator_present"] == 1)
    |
    (final_synthetic["can_leave_safely"] == 0)
    |
    (final_synthetic["medical_help"] == 1)
)

expected_risk = np.select(
    [expected_high, expected_medium],
    ["HIGH", "MEDIUM"],
    default="LOW"
)

risk_mismatches = (
    final_synthetic["immediate_safety_risk"]
    != expected_risk
).sum()

score_recomputed = sum(
    final_synthetic[col].astype(int) * weight
    for col, weight in DAWI_WEIGHTS.items()
)

score_mismatches = (
    final_synthetic["dawi_reference_score"]
    != score_recomputed
).sum()

print("Invalid immediate-threat records:", invalid_immediate)
print("Risk-label mismatches:", risk_mismatches)
print("DA-WI score mismatches:", score_mismatches)
print("Missing values:", final_synthetic.isna().sum().sum())

Invalid immediate-threat records: 0
Risk-label mismatches: 0
DA-WI score mismatches: 0
Missing values: 0


In [49]:
# ============================================================
# FINAL DATASET DISTRIBUTION
# ============================================================

print("=" * 80)
print("FINAL CONTROLLED SYNTHETIC DATASET")
print("=" * 80)

print("Shape:", final_synthetic.shape)

print("\nRisk distribution:")
display(
    final_synthetic[
        "immediate_safety_risk"
    ].value_counts(normalize=True)
)

print("\nSevere IPV reference distribution:")
display(
    final_synthetic[
        "severe_ipv_risk"
    ].value_counts(normalize=True)
)

print("\nDA-WI score:")
print(
    final_synthetic[
        "dawi_reference_score"
    ].describe()
)

print("\nApp-facing features:")
display(app_comparison)

FINAL CONTROLLED SYNTHETIC DATASET
Shape: (20000, 37)

Risk distribution:


immediate_safety_risk
LOW       0.65805
MEDIUM    0.32090
HIGH      0.02105
Name: proportion, dtype: float64


Severe IPV reference distribution:


severe_ipv_risk
0    0.9265
1    0.0735
Name: proportion, dtype: float64


DA-WI score:
count    20000.000000
mean         2.977100
std          2.730009
min          0.000000
25%          0.000000
50%          2.000000
75%          5.000000
max         22.000000
Name: dawi_reference_score, dtype: float64

App-facing features:


,feature,seed_rate,final_synthetic_rate,absolute_difference
0,safe_now,0.7038,0.86160,0.15780
1,perpetrator_present,0.2244,0.11145,0.11295
2,can_leave_safely,0.6936,0.83605,0.14245
3,medical_help,0.0621,0.05085,0.01125
4,contact_requested,0.2505,0.14525,0.10525


In [50]:
# ============================================================
# ACTUAL ON-DEVICE MODEL INPUTS
# ============================================================

MODEL_FEATURES = [
    "safe_now",
    "perpetrator_present",
    "can_leave_safely",
    "medical_help",
    "contact_requested"
]

print("Final on-device model features:")
print(MODEL_FEATURES)

display(
    final_synthetic[
        MODEL_FEATURES + [
            "immediate_safety_risk"
        ]
    ].head(15)
)

Final on-device model features:
['safe_now', 'perpetrator_present', 'can_leave_safely', 'medical_help', 'contact_requested']


,safe_now,perpetrator_present,can_leave_safely,medical_help,contact_requested,immediate_safety_risk
0,1,0,1,0,0,LOW
1,1,0,1,0,0,LOW
2,1,0,1,0,0,LOW
3,1,0,1,0,0,LOW
4,0,1,0,0,1,HIGH
5,1,0,1,0,0,LOW
6,1,0,1,0,0,LOW
7,1,0,1,0,0,LOW
8,1,0,1,0,0,LOW
9,1,0,1,0,0,LOW


In [51]:
FINAL_PATH = "../data/processed/final_synthetic_risk_dataset.csv"

final_synthetic.to_csv(
    FINAL_PATH,
    index=False
)

print("FINAL DATASET SAVED")
print("Path:", FINAL_PATH)
print("Shape:", final_synthetic.shape)

FINAL DATASET SAVED
Path: ../data/processed/final_synthetic_risk_dataset.csv
Shape: (20000, 37)
